# A-Polymer Update Strategy Matrix

This notebook runs a concrete experiment matrix for three constant-food A-polymer networks and four SSA/blended update configurations. The output layout is `outputs/<timestamp>/experiment_matrix/`, containing `test_config.csv/xlsx`, `test_result.csv/xlsx`, trajectories, and per-cell cProfile reports with the top 40 cumulative-time entries.

In [1]:
from pathlib import Path
import sys
import pandas as pd

COMPARE_DIR = Path.cwd()
if COMPARE_DIR.name != 'compare':
    COMPARE_DIR = Path(r'C:/Users/33973/Documents/New project/examples/compare')
sys.path.insert(0, str(COMPARE_DIR))

from experiment_matrix import (
    run_experiment_matrix,
    matrix_run_dir,
    write_config_files,
    load_config_dataframe,
)
from a_polymer_update_matrix import (
    DEFAULT_A_POLYMER_NETWORKS,
    create_a_polymer_update_config,
)


## Configuration Info

Edit this block to control the three networks, wall-clock budget, worker count, and blended parameters.

In [2]:
NETWORKS = list(DEFAULT_A_POLYMER_NETWORKS)
WALL_SECONDS = 60.0
MAX_STEPS = 100_000_000
SEED = 123
T_END = 260.0 # 模拟时间
WORKERS = 20
PROFILE_LIMIT = 40
OUTPUT_ROOT = COMPARE_DIR / 'outputs'
TIMESTAMP = None  # None creates a timestamped output directory

BLENDED_I1 = 50.0
BLENDED_I2 = 70.0
BLENDED_DT_CLE = 0.01
BLENDED_DT_MACRO = 0.1

config_info = pd.DataFrame(
    [
        ('networks', NETWORKS),
        ('wall_seconds', WALL_SECONDS),
        ('max_steps', MAX_STEPS),
        ('seed', SEED),
        ('t_end', T_END),
        ('workers', WORKERS),
        ('profile_limit', PROFILE_LIMIT),
        ('output_root', str(OUTPUT_ROOT)),
        ('timestamp', TIMESTAMP),
        ('blended_i1', BLENDED_I1),
        ('blended_i2', BLENDED_I2),
        ('blended_dt_cle', BLENDED_DT_CLE),
        ('blended_dt_macro', BLENDED_DT_MACRO),
    ],
    columns=['parameter', 'value'],
)
display(config_info)


,parameter,value
0,networks,"[polymer_a_len8_a8_catalyzes_a_constant_food, ..."
1,wall_seconds,60.0
2,max_steps,100000000
3,seed,123
4,t_end,260.0
5,workers,20
6,profile_limit,40
7,output_root,c:\Users\33973\Documents\New project\examples\...
8,timestamp,None
9,blended_i1,50.0


## Matrix

Rows are method configurations. Columns are network instances. Each cell is a JSON run specification consumed by `run_experiment_matrix`.

In [3]:
config = create_a_polymer_update_config(
    networks=NETWORKS,
    wall_seconds=WALL_SECONDS,
    max_steps=MAX_STEPS,
    seed=SEED,
    t_end=T_END,
    blended_i1=BLENDED_I1,
    blended_i2=BLENDED_I2,
    blended_dt_cle=BLENDED_DT_CLE,
    blended_dt_macro=BLENDED_DT_MACRO,
)

preview_dir = matrix_run_dir(OUTPUT_ROOT, 'a_polymer_config_preview')
preview_paths = write_config_files(config, preview_dir)
display(config)
preview_paths


,polymer_a_len8_a8_catalyzes_a_constant_food,polymer_a_len9_a9_catalyzes_a_constant_food,polymer_a_len10_a10_catalyzes_a_constant_food
method_config_id,,,
ssa,"{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""..."
blended_global_beta_global_propensity,"{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""..."
blended_local_beta_global_propensity,"{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""..."
blended_local_beta_local_propensity,"{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""..."


{'csv': WindowsPath('c:/Users/33973/Documents/New project/examples/compare/outputs/a_polymer_config_preview/experiment_matrix/test_config.csv'),
 'xlsx': WindowsPath('c:/Users/33973/Documents/New project/examples/compare/outputs/a_polymer_config_preview/experiment_matrix/test_config.xlsx')}

## Run And Results

This block executes the matrix. Every enabled cell writes a trajectory plus `outputs/<timestamp>/experiment_matrix/profiles/<config>__<network>__<method>.prof` and the corresponding `_top40.txt` report.

In [4]:
run_info = run_experiment_matrix(
    config,
    output_root=OUTPUT_ROOT,
    timestamp=TIMESTAMP,
    workers=WORKERS,
    profile=True,
    profile_limit=PROFILE_LIMIT,
)

result = run_info['result']
long_result = run_info['long_result']
display(result)

summary_columns = [
    'status',
    'config_id',
    'network',
    'method',
    'simulation_final_time',
    'n_events',
    'ssa_steps',
    'con_steps',
    'wall_runtime_seconds',
    'trajectory_path',
    'profile_report_path',
    'error',
]
display(long_result[[column for column in summary_columns if column in long_result.columns]])
print(f"run_dir: {run_info['run_dir']}")


[matrix] run_dir=c:\Users\33973\Documents\New project\examples\compare\outputs\20260810_135709\experiment_matrix
[matrix] enabled_tasks=12 workers=12 profile=True profile_limit=40
[matrix] status=ok config=blended_local_beta_local_propensity network=polymer_a_len8_a8_catalyzes_a_constant_food method=gillespie_cle_hybrid sim_time=20.157325948578126 steps=12942 events=12915 ssa_steps=12733 con_steps=182 stop=max_runtime_seconds wall=60.17418330000146 trajectory=c:\Users\33973\Documents\New project\examples\compare\outputs\20260810_135709\experiment_matrix\trajectories\blended_local_beta_local_propensity__polymer_a_len8_a8_catalyzes_a_constant_food__gillespie_cle_hybrid.npz profile=c:\Users\33973\Documents\New project\examples\compare\outputs\20260810_135709\experiment_matrix\profiles\blended_local_beta_local_propensity__polymer_a_len8_a8_catalyzes_a_constant_food__gillespie_cle_hybrid_top40.txt error=
[matrix] status=ok config=blended_local_beta_global_propensity network=polymer_a_len10_

,polymer_a_len8_a8_catalyzes_a_constant_food,polymer_a_len9_a9_catalyzes_a_constant_food,polymer_a_len10_a10_catalyzes_a_constant_food
method_config_id,,,
ssa,"{""con_steps"":0,""config_id"":""ssa"",""error"":"""",""f...","{""con_steps"":0,""config_id"":""ssa"",""error"":"""",""f...","{""con_steps"":0,""config_id"":""ssa"",""error"":"""",""f..."
blended_global_beta_global_propensity,"{""con_steps"":279,""config_id"":""blended_global_b...","{""con_steps"":234,""config_id"":""blended_global_b...","{""con_steps"":586,""config_id"":""blended_global_b..."
blended_local_beta_global_propensity,"{""con_steps"":182,""config_id"":""blended_local_be...","{""con_steps"":35,""config_id"":""blended_local_bet...","{""con_steps"":292,""config_id"":""blended_local_be..."
blended_local_beta_local_propensity,"{""con_steps"":182,""config_id"":""blended_local_be...","{""con_steps"":35,""config_id"":""blended_local_bet...","{""con_steps"":146,""config_id"":""blended_local_be..."


,status,config_id,network,method,simulation_final_time,n_events,ssa_steps,con_steps,wall_runtime_seconds,trajectory_path,profile_report_path,error
0,ok,blended_local_beta_local_propensity,polymer_a_len8_a8_catalyzes_a_constant_food,gillespie_cle_hybrid,20.157326,12915,12733,182,60.174183,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
1,ok,blended_local_beta_global_propensity,polymer_a_len10_a10_catalyzes_a_constant_food,gillespie_cle_hybrid,21.951298,12833,12541,292,60.181297,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
2,ok,blended_local_beta_local_propensity,polymer_a_len10_a10_catalyzes_a_constant_food,gillespie_cle_hybrid,21.953901,13124,12978,146,60.174364,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
3,ok,blended_local_beta_global_propensity,polymer_a_len9_a9_catalyzes_a_constant_food,gillespie_cle_hybrid,13.673789,13026,12991,35,60.215844,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
4,ok,ssa,polymer_a_len9_a9_catalyzes_a_constant_food,gillespie_ssa,2.901014,16401,16401,0,60.227149,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
5,ok,blended_local_beta_global_propensity,polymer_a_len8_a8_catalyzes_a_constant_food,gillespie_cle_hybrid,20.157326,12915,12733,182,60.189556,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
6,ok,blended_global_beta_global_propensity,polymer_a_len8_a8_catalyzes_a_constant_food,gillespie_cle_hybrid,20.166642,14696,14417,279,60.189964,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
7,ok,blended_local_beta_local_propensity,polymer_a_len9_a9_catalyzes_a_constant_food,gillespie_cle_hybrid,13.672758,12825,12790,35,60.198646,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
8,ok,blended_global_beta_global_propensity,polymer_a_len10_a10_catalyzes_a_constant_food,gillespie_cle_hybrid,21.957940,14317,13731,586,60.185765,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
9,ok,ssa,polymer_a_len8_a8_catalyzes_a_constant_food,gillespie_ssa,2.854468,15990,15990,0,60.166793,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,


run_dir: c:\Users\33973\Documents\New project\examples\compare\outputs\20260810_135709\experiment_matrix


## Load Existing Config

To run an edited config file later, load `test_config.csv` and pass it to `run_experiment_matrix`.

In [ ]:
# edited_config = load_config_dataframe(preview_paths['csv'])
# run_experiment_matrix(
#     edited_config,
#     output_root=OUTPUT_ROOT,
#     workers=4,
#     profile=True,
#     profile_limit=40,
# )
